# Fase 5d — Inserción de los chunks (con embeddings) en Elasticsearch

Carga, para cada una de las cuatro estrategias de chunking (Fase 3b), el JSON correspondiente ya enriquecido con sus embeddings (Fase 5c) y lo inserta de forma masiva (`helpers.bulk`) en su índice de Elasticsearch (creado en la Fase 5a), usando `insertar_json_en_indice`/`generar_acciones`.

Las celdas finales verifican el resultado: listan los índices del clúster y consultan un documento de ejemplo del índice de la estrategia Markdown para comprobar que el texto, los metadatos y la estructura de campos (incluidos los vectores) se han indexado correctamente.

In [1]:
import json
from elasticsearch import Elasticsearch, helpers

# 1. Conexión a tu Elasticsearch LOCAL
es = Elasticsearch("http://localhost:9200",
                   request_timeout=60,      # Esperamos hasta 60 segundos por respuesta
                   max_retries=3,           # Si falla, lo intenta 3 veces más
                   retry_on_timeout=True    # Permite reintentar si da timeout
                   )

print("Conectado a Elasticsearch local:", es.ping())

if not es.ping():
    raise ValueError("No se pudo conectar a Elasticsearch")

def insertar_json_en_indice(ruta_archivo, nombre_indice):
    """
    Carga un JSON de chunks con embeddings ya calculados y los inserta de
    forma masiva (bulk) en el índice de Elasticsearch indicado, usando el
    propio chunk_id como _id del documento (para que reinsertar el mismo
    archivo actualice en lugar de duplicar), y refresca el índice al
    terminar.
    """
    print(f"\nCargando datos de {ruta_archivo}...")
    
    # Abrimos el JSON
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
        
    print(f"Preparando {len(chunks)} documentos para el índice '{nombre_indice}'...")
    

    def generar_acciones(chunks, nombre_indice):
        """
        Generador que traduce cada chunk a la acción de bulk que espera la
        API de Elasticsearch (índice destino, id y documento fuente).
        """
        for chunk in chunks:
            yield {
                "_index": nombre_indice,
                "_id": chunk["chunk_id"],
                "_source": chunk
            }

    exito, errores = helpers.bulk(es, generar_acciones(chunks, nombre_indice), chunk_size=500, raise_on_error=False, stats_only=False)

    es.indices.refresh(index=nombre_indice)
    
    print(f"¡Completado! Documentos insertados: {exito}")
    if errores:
        print(f"Hubo {len(errores)} errores durante la inserción.")

# ==========================================
# EJECUCIÓN
# ==========================================

# Mapeamos qué archivo va a qué índice
base_path = "D:/Backup/Escritorio/CUARTO_CARRERA/TFG/resultados_chunking_embeddings"

archivos_a_indices = {
    f"{base_path}/chunks_estrategia_A_fixed_embeddings.json": "indice_estrategia_a_fixed",
    f"{base_path}/chunks_estrategia_B_markdown_embeddings.json": "indice_estrategia_b_markdown",
    f"{base_path}/chunks_estrategia_C_semantica_embeddings.json": "indice_estrategia_c_semantica",
    f"{base_path}/chunks_estrategia_D_jerarquico_embeddings.json": "indice_estrategia_d_jerarquico"
}
for archivo, indice in archivos_a_indices.items():
    try:
        insertar_json_en_indice(archivo, indice)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{archivo}'. Asegúrate de que está en la misma carpeta.")

Conectado a Elasticsearch local: True

Cargando datos de D:/Backup/Escritorio/CUARTO_CARRERA/TFG/resultados_chunking_embeddings/chunks_estrategia_A_fixed_embeddings.json...
Preparando 1654 documentos para el índice 'indice_estrategia_a_fixed'...
¡Completado! Documentos insertados: 1654

Cargando datos de D:/Backup/Escritorio/CUARTO_CARRERA/TFG/resultados_chunking_embeddings/chunks_estrategia_B_markdown_embeddings.json...
Preparando 2031 documentos para el índice 'indice_estrategia_b_markdown'...
¡Completado! Documentos insertados: 2031

Cargando datos de D:/Backup/Escritorio/CUARTO_CARRERA/TFG/resultados_chunking_embeddings/chunks_estrategia_C_semantica_embeddings.json...
Preparando 1881 documentos para el índice 'indice_estrategia_c_semantica'...
¡Completado! Documentos insertados: 1881

Cargando datos de D:/Backup/Escritorio/CUARTO_CARRERA/TFG/resultados_chunking_embeddings/chunks_estrategia_D_jerarquico_embeddings.json...
Preparando 3330 documentos para el índice 'indice_estrategia_

In [2]:
# Muestra una tabla en texto con todos los índices del clúster
print(es.cat.indices(v=True))

health status index                          uuid                   pri rep docs.count docs.deleted store.size pri.store.size dataset.size
yellow open   indice_estrategia_a_fixed      ogWoF8VCQr-mkozaKjbLjg   1   1       1654            0     16.9mb         16.9mb       16.9mb
yellow open   indice_estrategia_d_jerarquico Fw3jz5RWSg2OQrlrvWJ8nw   1   1       3330            0      4.4mb          4.4mb        4.4mb
yellow open   indice_estrategia_c_semantica  QGrNk11_RA-p7GDCoWFP9Q   1   1       1881            0       19mb           19mb         19mb
yellow open   indice_estrategia_b_markdown   TZHDlG80TMeIWpykBp2Emw   1   1       2031            0     20.5mb         20.5mb       20.5mb



In [3]:
indice_prueba = "indice_estrategia_b_markdown"
resultado = es.search(index=indice_prueba, size=1)

# Verificamos si hay resultados
if resultado['hits']['total']['value'] > 0:
    documento = resultado['hits']['hits'][0]['_source']
    
    print(f"--- REVISIÓN DEL ÍNDICE: {indice_prueba.upper()} ---")
    
    # Comprobación de Texto y Metadatos
    print("\nTexto del chunk (primeros 150 caracteres):")
    print(f"   -> {documento.get('text', 'No hay texto')[:150]}...")
    
    if 'metadata' in documento:
        print("\nMetadatos guardados:")
        for clave, valor in documento['metadata'].items():
            print(f"   -> {clave}: {valor}")
else:
    print("El índice está vacío.")

--- REVISIÓN DEL ÍNDICE: INDICE_ESTRATEGIA_B_MARKDOWN ---

Texto del chunk (primeros 150 caracteres):
   -> ## Justificación

En el año 2014, el Ministerio de Sanidad, Servicios Sociales e Igualdad (MSSSI) puso en marcha el Plan Nacional frente a la Resistec...

Metadatos guardados:
   -> source: 1181_Guía hospitalaria de terapéutica antibiótica en adultos.pdf
   -> domain: protocolo_clinico_murcia
   -> type: text_markdown
   -> titulo: Guía hospitalaria de terapéutica antibiótica en adultos
   -> autores: Parra-Hidalgo, Pedro, Alcalde-Encinas, Mar, Bernal-Morell, Enrique, Blázquez-Garrido, Rosa, Calle-Urra, José-Eduardo, Cobos-Trigueros, Nazaret, Espinosa-Parra, Francisco-Javier, García-Vázquez, Elisa, Garre-García, Adriana, Guerrero, Carmen, Jimeno-Almazán, Amaya, Moral-Escudero, Encarnación, Olmos-Jiménez, Raquel, Peláez-Ballesta, Ana-Isabel, Sánchez-Serrano, Adriana, Sarabia-Marco, Francisco-de-Asís, Yagüe-Guirao, Genoveva
   -> fecha_publicacion: 2020-12
   -> tema: Antibacterian

In [4]:
# Esta celda continúa la anterior y reutiliza la variable 'documento'
# (el primer resultado obtenido en la celda de revisión del índice).

print("\nCLAVES PRINCIPALES DEL DOCUMENTO:")
print(list(documento.keys()))


CLAVES PRINCIPALES DEL DOCUMENTO:
['chunk_id', 'text', 'metadata']
